# Practice Lab 5.6 - Building production MCP servers in Python (WORKED solutions)

**Course:** Netsetos FDE Cohort 1 - Module 5 - Lesson 5.6 - Practice Lab

This notebook is the **worked-solution companion** to `practice-lab-5.6.html` (9 exercises).
Each exercise below repeats the prompt and success criteria, then shows a runnable worked
solution. Run it top to bottom.

- **No API key needed.** Everything runs offline against synthetic AcmeCorp CRM data - the only
  dependencies are `fastmcp>=3.0` and `pydantic>=2.0` (Exercise 8 also uses `pytest` +
  `pytest-asyncio`). There are no anthropic/openai calls in this lesson, so there is no keyless
  API mock to set up.
- The pure-policy cells (Ex 3, Ex 6) run and print with no MCP host at all; the MCP tool /
  resource / prompt / middleware cells define the server objects a host (Claude Desktop, Cursor,
  the OpenAI Agents SDK) would spawn over stdio.
- Production context: AcmeCorp support - the M5 mini-capstone deliverable #1. Synthetic IDs only.

In [ ]:
!pip install --quiet fastmcp pydantic pytest pytest-asyncio

---
## Exercise 1 - The 30-line baseline server

*One tool, one docstring, one type hint - and it boots*

**Task.** Complete `get_customer(customer_id: str) -> dict` on a FastMCP server named `acmecorp-crm`. Return the customer record on a hit and a structured `{"error": True, "code": "CUSTOMER_NOT_FOUND"}` on a miss. Never raise. Add the docstring - it becomes the tool description the LLM reads.

**Success criteria**
- boots over stdio with no import error (4)
- docstring present and surfaces as the tool description (4)
- miss returns the structured dict rather than raising or returning `None` (4)

In [ ]:
# Worked solution - the 30-line baseline: one tool, a docstring, a structured miss.
from fastmcp import FastMCP

mcp = FastMCP(name="acmecorp-crm")
_CUSTOMERS = {"c_001": {"customer_id": "c_001", "name": "Aisha Khan", "tier": "Pro"}}

@mcp.tool()
def get_customer(customer_id: str) -> dict:
    """Look up an AcmeCorp customer by ID.

    The LLM reads THIS docstring as the tool description, so name the input
    and the output shape in plain language.

    Returns the record on a hit, or a structured error object
    {"error": True, "code": "CUSTOMER_NOT_FOUND", "customer_id": ...} on a miss.
    Never raises - a structured error keeps the agent loop alive.
    """
    record = _CUSTOMERS.get(customer_id)
    if record is None:
        return {"error": True, "code": "CUSTOMER_NOT_FOUND", "customer_id": customer_id}
    return record

if __name__ == "__main__":
    mcp.run()  # stdio is the default

# -> tools/list advertises one tool whose inputSchema requires customer_id: string
#    and whose description is this docstring's first sentence.
# -> get_customer("c_001") returns the record;
#    get_customer("c_FAKE") returns {"error": True, "code": "CUSTOMER_NOT_FOUND", ...}

---
## Exercise 2 - Tool schema the LLM follows on the first attempt

*Pydantic enum + docstring so the model routes correctly*

**Task.** Add `list_orders(customer_id, status: OrderStatus | None = None, limit: int = 20)`. Use a `str, Enum` for `OrderStatus` (`pending`, `paid`, `pending_refund`) so FastMCP ships an enum-constrained schema. Write a docstring that tells the model "use this to list a customer's orders, not to look up one customer".

**Success criteria**
- `status` is a real enum in the emitted schema, not a free string (5)
- docstring disambiguates from `get_customer` so routing is deterministic (4)
- `limit` defaulted and honored (3)

In [ ]:
# Worked solution - an enum-constrained schema so the model routes on the first try.
from enum import Enum
from fastmcp import FastMCP

mcp = FastMCP(name="acmecorp-crm")

class OrderStatus(str, Enum):
    PENDING = "pending"
    PAID = "paid"
    PENDING_REFUND = "pending_refund"

_ORDERS = {
    "o_4823": {"order_id": "o_4823", "customer_id": "c_001", "status": "pending_refund"},
    "o_5020": {"order_id": "o_5020", "customer_id": "c_002", "status": "paid"},
}

@mcp.tool()
def list_orders(customer_id: str, status: OrderStatus | None = None, limit: int = 20) -> list[dict]:
    """List a customer's orders. Use this to list orders, NOT to look up one customer
    (that is get_customer). Filter by status when provided; cap results at limit.

    Args:
        customer_id: AcmeCorp customer identifier.
        status: optional OrderStatus filter (pending | paid | pending_refund).
        limit: max rows to return (default 20, capped at 100).
    """
    limit = max(1, min(limit, 100))
    rows = [o for o in _ORDERS.values()
            if o["customer_id"] == customer_id
            and (status is None or o["status"] == status.value)]
    return rows[:limit]

# -> the emitted inputSchema constrains status to {pending, paid, pending_refund};
#    an unknown enum value returns InvalidParams from the protocol layer (schema-as-contract).
# -> the docstring's first line disambiguates from get_customer, so routing is deterministic.
# -> limit defaults to 20 and is honored (and clamped to 1..100).

---
## Exercise 3 - Structured errors, never raise across the boundary

*check_refund_eligibility returns a failure shape, not an exception*

**Task.** Implement `check_refund_eligibility(order_id) -> dict` against policy v2.3. Return `{"eligible", "reason", "policy_version", "order_id", "error_code"}` for every path: unknown order (`ORDER_NOT_FOUND`), wrong status (`WRONG_STATUS`), outside the 30-day window (`OUTSIDE_WINDOW`), and the eligible case (`error_code=None`). No `raise` anywhere. Factor the policy body into a plain helper `_eligibility(order_id) -> dict` and have the tool call it - Exercise 4 reuses that helper for the batch path.

**Success criteria**
- all four paths return the five-key shape (8)
- `policy_version` stamped on every branch (3)
- zero `raise` across the boundary and the pure helper is reusable (4)

In [ ]:
# Worked solution - policy v2.3 factored into a PURE helper; the tool is a one-liner over it.
from datetime import datetime, timezone

_NOW = datetime(2026, 5, 21, 12, 0, 0, tzinfo=timezone.utc)
REFUND_WINDOW_DAYS = 30
POLICY_VERSION = "v2.3"

# order_id -> (status, placed_days_ago)  (synthetic; Ex 4 reuses this map + the helper)
_ORDERS = {
    "o_4823": ("pending_refund", 21),   # eligible
    "o_1200": ("pending_refund", 45),   # OUTSIDE_WINDOW
    "o_5590": ("pending_refund", 3),    # eligible
    "o_7001": ("paid", 5),              # WRONG_STATUS
}

def _eligibility(order_id: str) -> dict:
    """Pure policy logic (v2.3), no MCP knowledge. Never raises. Returns the 5-key shape."""
    order = _ORDERS.get(order_id)
    if order is None:
        return {"eligible": False, "reason": "order not found in AcmeCorp CRM",
                "policy_version": POLICY_VERSION, "order_id": order_id,
                "error_code": "ORDER_NOT_FOUND"}
    status, age_days = order
    if status != "pending_refund":
        return {"eligible": False,
                "reason": f"order status is '{status}'; only pending_refund may be processed",
                "policy_version": POLICY_VERSION, "order_id": order_id,
                "error_code": "WRONG_STATUS"}
    if age_days > REFUND_WINDOW_DAYS:
        return {"eligible": False,
                "reason": f"order placed {age_days} days ago; outside the {REFUND_WINDOW_DAYS}-day window",
                "policy_version": POLICY_VERSION, "order_id": order_id,
                "error_code": "OUTSIDE_WINDOW"}
    return {"eligible": True,
            "reason": f"within {REFUND_WINDOW_DAYS}-day window; order placed {age_days} days ago",
            "policy_version": POLICY_VERSION, "order_id": order_id, "error_code": None}

# The MCP tool just calls the pure helper (registered only if fastmcp is importable):
try:
    from fastmcp import FastMCP
    mcp = FastMCP(name="acmecorp-crm")

    @mcp.tool()
    def check_refund_eligibility(order_id: str) -> dict:
        """Determine refund eligibility (policy v2.3). Never raises."""
        return _eligibility(order_id)
except Exception:
    pass  # no fastmcp -> the pure helper below is still the graded logic and still runs

# Genuinely runnable: exercise all four policy paths through the pure helper.
for oid in ["o_4823", "o_1200", "o_7001", "o_DOES_NOT_EXIST"]:
    r = _eligibility(oid)
    print(f"{oid:18s} eligible={str(r['eligible']):5s} error_code={r['error_code']}")

---
## Exercise 4 - The Context object: structured logging + progress on a batch tool

*batch_check_refunds streams progress and logs every decision via ctx*

**Task.** AcmeCorp support runs weekly refund sweeps - an agent hands the server a list of order IDs and expects one call, a progress bar, and an audit line per order. This is the primitive most first MCP servers skip: the injected `Context`. Add a parameter typed `ctx: Context` and FastMCP injects the live session; `await ctx.report_progress(...)` drives the host's progress UI and `await ctx.info(...)` emits a structured log the host and your observability pipeline both see. Because you `await` the context, the tool must be `async def`.

**Success criteria**
- tool is `async` and takes `ctx: Context` that FastMCP injects (5)
- `report_progress` fires once per order with a correct `progress/total` (5)
- one structured `ctx.info` line per order carrying `order_id` + `error_code`, plus start/summary lines (6)
- a bad ID becomes a result row, never an exception, and the return aggregates `total/eligible/results` (4)

In [ ]:
# Worked solution - the injected Context: stream progress + one audit line per order.
from fastmcp import FastMCP, Context

# mcp and _eligibility come from the Exercise 3 cell when you run top to bottom.

@mcp.tool()
async def batch_check_refunds(order_ids: list[str], ctx: Context) -> dict:
    """Check refund eligibility across many orders in one call.

    Streams progress so the host can render a bar, and logs one structured line per
    order for the audit trail. Reuses _eligibility from Ex 3. Never raises - a bad ID
    becomes a row with an error_code, not a crash.
    """
    total = len(order_ids)
    await ctx.info(f"batch_check_refunds start: {total} orders")
    results, eligible = [], 0
    for i, oid in enumerate(order_ids, start=1):
        row = _eligibility(oid)                      # reuse the Ex 3 helper
        await ctx.report_progress(progress=i, total=total)
        await ctx.info(f"{oid} -> eligible={row['eligible']} error_code={row['error_code']}")
        results.append(row)
        if row["eligible"]:
            eligible += 1
    await ctx.info(f"batch_check_refunds done: {eligible}/{total} eligible")
    return {"total": total, "eligible": eligible, "results": results}

# -> calling with ["o_4823", "o_1200", "o_DOES_NOT_EXIST", "o_5590"] emits:
#    [info]     batch_check_refunds start: 4 orders
#    [progress] 1/4   [info] o_4823 -> eligible=True error_code=None
#    [progress] 2/4   [info] o_1200 -> eligible=False error_code=OUTSIDE_WINDOW
#    [progress] 3/4   [info] o_DOES_NOT_EXIST -> eligible=False error_code=ORDER_NOT_FOUND
#    [progress] 4/4   [info] o_5590 -> eligible=True error_code=None
#    [info]     batch_check_refunds done: 2/4 eligible
#    return -> {"total": 4, "eligible": 2, "results": [ ...4 rows... ]}
# In a test you capture these via Client(server, progress_handler=..., log_handler=...).

---
## Exercise 5 - Resource and prompt, the under-taught primitives

*An addressable resource and a host-injectable prompt template*

**Task.** Add `@mcp.resource("customer-context://customers/{customer_id}")` returning a compact context blob (`tier`, `ltv_usd`, `open_tickets`, `churn_risk`, `available`). Add `@mcp.prompt() escalation_template(customer_id, issue_summary)` that renders one escalation line. Handle the unknown-customer case in both without raising.

**Success criteria**
- resource URI template resolves `{customer_id}` and returns the blob (6)
- prompt renders a single clean escalation line (5)
- unknown customer returns `available: false` / a safe placeholder rather than raising (4)

In [ ]:
# Worked solution - the two under-taught primitives: an addressable resource + a prompt.
# mcp comes from an earlier cell; POLICY_VERSION from Ex 3.
_CUSTOMERS_FULL = {
    "c_001": {"customer_id": "c_001", "name": "Aisha Khan", "tier": "Pro",
              "ltv_usd": 4820.0, "open_tickets": 2, "churn_risk": "low"},
}

@mcp.resource("customer-context://customers/{customer_id}")
def customer_context(customer_id: str) -> dict:
    """Compact customer-context blob for prepending to an agent's working memory.

    URI: customer-context://customers/{customer_id}. Unknown id -> available False,
    never raises.
    """
    c = _CUSTOMERS_FULL.get(customer_id)
    if c is None:
        return {"customer_id": customer_id, "available": False, "reason": "customer not in CRM"}
    return {"customer_id": c["customer_id"], "tier": c["tier"], "ltv_usd": c["ltv_usd"],
            "open_tickets": c["open_tickets"], "churn_risk": c["churn_risk"], "available": True}

@mcp.prompt()
def escalation_template(customer_id: str, issue_summary: str) -> str:
    """Render a single escalation line for the support team. Unknown id -> safe placeholder."""
    c = _CUSTOMERS_FULL.get(customer_id)
    if c is None:
        return f"Escalation for unknown customer '{customer_id}'. Issue: {issue_summary}. Verify the ID first."
    return (f"Customer {c['customer_id']} ({c['name']}, {c['tier']} tier, "
            f"LTV ${c['ltv_usd']:.2f}, churn risk {c['churn_risk']}) reports: {issue_summary}.")

# -> fetching customer-context://customers/c_001 returns
#    {"customer_id": "c_001", "tier": "Pro", "ltv_usd": 4820.0, ..., "available": True}
# -> escalation_template("c_001", "refund delay on o_4823") renders
#    "Customer c_001 (Aisha Khan, Pro tier, LTV $4820.00, churn risk low) reports: refund delay on o_4823."

---
## Exercise 6 - Version discipline and additive change

*Bump semver correctly and keep old clients working*

**Task.** Declare `FastMCP(name="acmecorp-crm", version="2.1.0", instructions=...)`. Then extend `list_orders` with a new optional `tier: CustomerTier | None = None` filter (a minor bump - old clients that never pass `tier` keep working). Write a one-line rule table classifying: add a tool, add an optional arg, rename a required arg, remove a tool.

**Success criteria**
- the four-row classification is correct with a one-line why each (8)
- the `tier` filter is genuinely optional so a client that omits it is unaffected (5)
- declared version surfaces in `info` (3)

In [ ]:
# Worked solution - the semver classification (runnable) + the additive optional tier filter.
SEMVER_RULES = {
    "add a new tool":             ("minor", "additive - old clients keep working"),
    "add an optional argument":   ("minor", "additive - omitting it is unchanged behaviour"),
    "rename a required argument":  ("major", "breaking - old callers hit InvalidParams"),
    "remove a tool":              ("major", "breaking - callers of that tool now fail"),
}
for change, (bump, why) in SEMVER_RULES.items():
    print(f"{change:26s} -> {bump:5s} ({why})")

# The version is declared on the server; the new tier filter is a MINOR (additive) bump.
try:
    from enum import Enum
    from fastmcp import FastMCP

    class CustomerTier(str, Enum):
        FREE = "Free"
        PRO = "Pro"
        BUSINESS = "Business"
        ENTERPRISE = "Enterprise"

    class OrderStatus(str, Enum):
        PENDING = "pending"
        PAID = "paid"
        PENDING_REFUND = "pending_refund"

    mcp = FastMCP(
        name="acmecorp-crm", version="2.1.0",
        instructions="AcmeCorp CRM v2.1.0 - added an optional 'tier' filter to list_orders (additive).",
    )

    @mcp.tool()
    def list_orders(customer_id: str, status: OrderStatus | None = None,
                    tier: CustomerTier | None = None, limit: int = 20) -> list[dict]:
        """List a customer's orders. New in v2.1: an optional 'tier' filter.
        Old clients that never pass tier keep working unchanged."""
        ...
except Exception:
    pass

# -> add tool: minor | add optional arg: minor | rename required arg: major | remove tool: major
# -> the server info response reports version "2.1.0"; a client that omits tier is unaffected.

---
## Exercise 7 - Observability: instrument every tool call

*Per-tool latency, error_code counts, and tools/call volume as structured logs*

**Task.** When an AcmeCorp refund flow misfires at 2am, nobody wants to grep raw stdout. Add a FastMCP middleware that wraps every `tools/call`, times it, reads the tool's structured `error_code` out of the result, and emits exactly one JSON line per call through `ctx.info` - the same channel Exercise 4 used. Keep two in-process counters (call volume per tool, error-code frequency) so you can print a snapshot. This is the seam Module 6 plugs OpenTelemetry / Langfuse into: you produce one parseable event per call now, and M6 swaps the JSON print for a real span/trace exporter.

**Success criteria**
- middleware wraps every `tools/call` and measures latency around `call_next` (6)
- `error_code` read from the structured result, `None` handled (5)
- both counters increment correctly and the snapshot is right (5)
- exactly one JSON `ctx.info` line per call, and a one-sentence note on where it ships (OTel/Langfuse, M6) (4)

In [ ]:
# Worked solution - one JSON line per tools/call: latency + error_code + call volume.
import time, json
from collections import Counter
from fastmcp.server.middleware import Middleware, MiddlewareContext

_TOOL_CALLS = Counter()      # tools/call volume per tool
_ERROR_CODES = Counter()     # error_code frequency across all tools

class ObservabilityMiddleware(Middleware):
    async def on_call_tool(self, context: MiddlewareContext, call_next):
        tool = context.message.name
        t0 = time.perf_counter()
        result = await call_next(context)                 # run the tool
        latency_ms = round((time.perf_counter() - t0) * 1000, 1)
        payload = result.structured_content or {}         # FastMCP 3.x
        error_code = payload.get("error_code")            # may be None
        _TOOL_CALLS[tool] += 1
        if error_code:
            _ERROR_CODES[error_code] += 1
        ctx = context.fastmcp_context
        if ctx:
            await ctx.info(json.dumps({
                "tool": tool, "latency_ms": latency_ms,
                "error_code": error_code, "calls": _TOOL_CALLS[tool]}))
        return result

mcp.add_middleware(ObservabilityMiddleware())

# -> one line per call, e.g.
#    {"tool": "check_refund_eligibility", "latency_ms": 1.3, "error_code": "OUTSIDE_WINDOW", "calls": 1}
#    snapshot: _TOOL_CALLS = {'get_customer': 1, 'check_refund_eligibility': 2}
#              _ERROR_CODES = {'OUTSIDE_WINDOW': 1}
# In Module 6 you swap the json.dumps print for an OpenTelemetry span (name=tool,
# duration=latency_ms, error_code as a span attribute) or a Langfuse trace.

---
## Exercise 8 - pytest with the in-memory client (the CI gate)

*Prove the error path stays structured, in under a second*

**Task.** Write a pytest suite using FastMCP's in-memory `Client(server)` - no subprocess, no socket. Cover: lists the tools, happy-path lookup, unknown-customer structured error, refund inside vs outside the window, unknown-order structured error, the batch tool's progress/log handlers fire, resource fetch, prompt render, and a schema-violation returning `InvalidParams`. Assert the critical test: an unknown order returns `eligible=False, error_code="ORDER_NOT_FOUND"`, NOT a raised exception.

**Success criteria**
- the unknown-order test proves structured (not raised) failure (6)
- coverage spans tools/list, happy path, both refund-window branches, resource, prompt, and an `InvalidParams` schema violation (8)
- the batch progress/log handler test passes (3)
- suite runs under a second in-memory (3)

In [ ]:
# Worked solution - the in-memory pytest gate. Save the Ex 2 server as acmecorp_server.py first.
import pytest
from fastmcp import Client
from acmecorp_server import mcp as server   # the Ex 2 server object

@pytest.mark.asyncio
async def test_unknown_order_returns_structured_error():
    async with Client(server) as client:
        result = await client.call_tool(
            "check_refund_eligibility", {"order_id": "o_DOES_NOT_EXIST"})
        payload = result.data
        assert payload["eligible"] is False
        assert payload["error_code"] == "ORDER_NOT_FOUND"   # structured, NOT a raised exception

@pytest.mark.asyncio
async def test_batch_reports_progress():
    seen = []
    async def on_progress(progress, total, message):
        seen.append((progress, total))
    async with Client(server, progress_handler=on_progress) as client:
        await client.call_tool("batch_check_refunds",
                               {"order_ids": ["o_4823", "o_1200"]})
        assert seen == [(1, 2), (2, 2)]

@pytest.mark.asyncio
async def test_lists_tools_and_happy_path():
    async with Client(server) as client:
        names = {t.name for t in await client.list_tools()}
        assert {"get_customer", "list_orders", "check_refund_eligibility"} <= names
        rec = (await client.call_tool("get_customer", {"customer_id": "c_001"})).data
        assert rec["name"] == "Aisha Khan"

@pytest.mark.asyncio
async def test_schema_violation_returns_invalid_params():
    async with Client(server) as client:
        with pytest.raises(Exception) as exc:
            await client.call_tool("list_orders",
                                   {"customer_id": "c_001", "limit": "not-an-int"})
        low = str(exc.value).lower()
        assert "limit" in low or "invalid" in low or "valid" in low

# -> pytest -xvs runs 10+ tests in ~0.5s (a subprocess + stdio equivalent would take ~8s).
#    The unknown-order test proves a STRUCTURED failure, not a raised exception.

---
## Exercise 9 - Package it: wheel, Docker, and Claude Desktop wiring

*Ship the server the way a customer actually installs it*

**Task.** A server that only runs from your checkout is a demo, not a deliverable. Package `acmecorp-crm` three ways: (a) a wheel with a `console_scripts` entry point (the L2.6 pattern) so `acmecorp-crm` is a command on PATH; (b) a multi-stage `uv` Dockerfile that builds the wheel in one stage and ships a slim runtime; (c) a `claude_desktop_config.json` block wiring the stdio transport so all tools appear in Claude Desktop's tools menu after a restart.

**Success criteria**
- `console_scripts` entry point resolves to `main()` and the wheel installs a working `acmecorp-crm` command (7)
- Dockerfile is genuinely multi-stage - build tooling absent from the runtime image, non-root user, `-i`/stdio ENTRYPOINT (7)
- `claude_desktop_config.json` is valid and launches the server so tools appear (4)
- a one-line note that stdout is protocol-only, logs go to stderr (2)

In [ ]:
# Worked solution (Python part) - the console_scripts entry point main().
# stdout carries JSON-RPC frames ONLY; every log line goes to stderr or the client desyncs.
def main() -> None:
    """stdio by default; --http switches to streamable HTTP for remote deployment."""
    import sys
    if "--http" in sys.argv:
        mcp.run(transport="http", host="0.0.0.0", port=8000)
    else:
        mcp.run()  # stdio - exactly what Claude Desktop spawns

if __name__ == "__main__":
    main()

# -> `uv build` produces dist/acmecorp_crm-2.1.0-py3-none-any.whl
# -> a fresh `pip install` of that wheel puts `acmecorp-crm` on PATH, speaking MCP over stdio.
# -> the packaging files (pyproject.toml, Dockerfile, claude_desktop_config.json) are in the
#    reference cell below - they are config, not runnable Python.

**Exercise 9 packaging files** (config, not runnable Python - shown filled in):

`pyproject.toml`
```toml
[project]
name = "acmecorp-crm"
version = "2.1.0"
requires-python = ">=3.11"
dependencies = ["fastmcp>=3.0", "pydantic>=2.0"]

[project.scripts]
acmecorp-crm = "acmecorp_crm.server:main"   # the L2.6 console-entry-point pattern

[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"
```

`acmecorp_crm/server.py` main() is the code cell above.

`Dockerfile`
```dockerfile
# multi-stage: uv builds the wheel in one stage, the runtime image stays slim
FROM ghcr.io/astral-sh/uv:python3.12-bookworm-slim AS build
WORKDIR /app
COPY pyproject.toml uv.lock ./
RUN uv sync --frozen --no-dev
COPY . .
RUN uv build --wheel -o /dist

FROM python:3.12-slim AS runtime
COPY --from=build /dist/*.whl /tmp/
RUN pip install --no-cache-dir /tmp/*.whl && useradd -m app
USER app
ENTRYPOINT ["acmecorp-crm"]   # stdio; run the container with -i
```

`claude_desktop_config.json`
```json
{
  "mcpServers": {
    "acmecorp-crm": {
      "command": "uvx",
      "args": ["--from", "acmecorp-crm", "acmecorp-crm"]
    }
  }
}
// Docker form instead:
//   "command": "docker",
//   "args": ["run", "-i", "--rm", "acmecorp/crm-mcp:2.1.0"]
```

---
## Grading rubric (150 points across 9 exercises)

| # | Criterion | Points | What full marks looks like |
|---|---|---|---|
| 1 | Baseline server | 12 | One tool boots over stdio; docstring becomes the description; miss returns a structured error, not an exception |
| 2 | Tool schema + enum | 12 | `list_orders` ships an enum-constrained `status`; docstring routes the model away from `get_customer` |
| 3 | Structured errors | 15 | All four refund paths return the failure shape via a reusable `_eligibility` helper; no `raise` across the boundary |
| 4 | Context: logging + progress | 20 | Async `batch_check_refunds` takes injected `ctx`; `report_progress` per order; one structured `ctx.info` line per order; bad IDs become rows |
| 5 | Resource + prompt | 15 | Resource URI returns the context blob; prompt renders the escalation line; both handle unknown customers |
| 6 | Version discipline | 16 | semver classification correct; the `tier` filter is additive; `info` reports the declared version |
| 7 | Observability | 20 | Middleware logs per-tool latency + error_code + call volume as one JSON line each; states it ships to OTel/Langfuse (M6) |
| 8 | pytest in-memory gate | 20 | 10+ tests green in under a second; unknown-order test proves structured failure; batch progress handler asserted |
| 9 | Packaging + wiring | 20 | Wheel with `console_scripts` entry point; multi-stage uv Dockerfile; valid `claude_desktop_config.json` shows all tools |

Pass mark: 105/150 (70%). A distinction (135+) swaps Exercise 7's `json.dumps` print for a real OpenTelemetry span exporter wired to a local Collector (or a Langfuse trace) as a Module 6 preview, adds a GitHub Actions workflow that runs the Exercise 8 pytest gate and builds the Exercise 9 wheel on every PR, and documents why this customer-specific CRM connector belongs in AcmeCorp's internal MCP registry, not the public GitHub MCP registry.

**Feedback SLA** - submit your solution in the cohort Discord `#practice-labs` channel. Rubric-graded feedback is returned within **24 hours** (Plus-tier mentor review); self-paced learners get this worked notebook + the rubric to self-assess right away.